# 21 · Batch scoring: from a registered model to risk, alerts and KPIs

**Who this is for:** you, after notebook 20 (training and the registry).
**Runs on:** Databricks serverless (costs credits: keep it short, detach when done). Run the `score` job first, so the Gold tables exist (staging runs it after every merge; prod via **prod-run** with *Score* ticked).

**By the end you will be able to:**
- explain what the `score` job does and which model version it uses (`@champion` in prod, the newest in dev and staging)
- explain why the alert threshold comes from training, not from the scoring run
- read `gold.risk_scores`, `alerts_fact`, `escalations_fact` and `site_kpis_hourly`
- say why an alert's outcome stays empty for 12 hours
- check that the model really catches more escalations than NEWS2, from the tables

The job is `databricks/resources/score.job.yml`; the logic is `heavden.ml.batch_scoring`, the same functions that build the app's demo snapshot (`tests/test_batch_scoring.py`). A serving endpoint (one patient at a time, for the app's what-if) comes later in this notebook series.

## 1. Setup

In [ ]:
dbutils.widgets.text("catalog", "heavden_prod")
CATALOG = dbutils.widgets.get("catalog")
print(CATALOG)

## 2. Which model scored, at which threshold

`risk_scores.model_version` records the version. The threshold is fixed **at training** on validation, as the lowest risk that keeps alerts within about 2 per nurse per 12-hour shift; the scoring job reads it from the model's MLflow run (`valid_threshold`). Refitting it at scoring time would move the alert budget whenever the data changed.

In [ ]:
display(
    spark.sql(f"""
    SELECT model_version, count(*) AS hours,
           min(prediction_ts) AS first_hour, max(prediction_ts) AS last_hour,
           round(avg(CASE WHEN risk_band = 'High' THEN 1 ELSE 0 END) * 100, 1) AS pct_high_hours,
           round(min(CASE WHEN risk_band = 'High' THEN risk END), 5) AS alert_threshold
    FROM {CATALOG}.gold.risk_scores GROUP BY model_version
""")
)

## 3. One patient's hours, with the reasons

`top_factors` holds the 3 biggest reasons for each score (grouped SHAP contributions, in log-odds), as JSON. This is what the app's patient page and the assistant's "why is this patient high risk?" show.

In [ ]:
patient = spark.sql(f"""
    SELECT encounter_id FROM {CATALOG}.gold.alerts_fact ORDER BY risk DESC LIMIT 1
""").first()[0]
display(
    spark.sql(f"""
    SELECT prediction_ts, round(risk, 4) AS risk, risk_band, news2_total, top_factors
    FROM {CATALOG}.gold.risk_scores WHERE encounter_id = '{patient}' ORDER BY prediction_ts
""")
)

## 4. Alerts, and why their outcome waits 12 hours

An alert = a patient **entering** High; the same patient can't alert again for 6 hours (alarm-fatigue suppression). `escalated_within_6h` is filled in once known: the 6-hour window must have passed **and** escalations are recorded about 6 hours late, so a "no" is final 12 hours after the alert. A recorded escalation settles a "yes" straight away. Same rule as the training labels.

In [ ]:
display(
    spark.sql(f"""
    SELECT CASE WHEN escalated_within_6h IS NULL THEN 'not known yet'
                WHEN escalated_within_6h = 1 THEN 'escalated' ELSE 'no escalation' END AS outcome,
           count(*) AS alerts, round(avg(hours_to_escalation), 1) AS mean_hours_warning
    FROM {CATALOG}.gold.alerts_fact GROUP BY 1 ORDER BY 1
""")
)

## 5. Did the model flag escalations in advance? And NEWS2?

`escalations_fact.flagged_6h_before` says whether the patient was High at any hour in the 6 hours before. For a fair comparison, flag NEWS2 at its own threshold with the same alert budget; NEWS2 is a whole number, so ≥ 5 (the clinical "urgent response" trigger) is a reasonable stand-in here.

In [ ]:
display(
    spark.sql(f"""
    WITH before AS (
      SELECT e.encounter_id, e.event_ts, e.flagged_6h_before,
             max(CASE WHEN r.news2_total >= 5 THEN 1 ELSE 0 END) AS news2_5_before
      FROM {CATALOG}.gold.escalations_fact e
      JOIN {CATALOG}.gold.risk_scores r
        ON r.encounter_id = e.encounter_id
       AND r.prediction_ts >= e.event_ts - INTERVAL 6 HOURS AND r.prediction_ts < e.event_ts
      GROUP BY ALL)
    SELECT count(*) AS escalations,
           sum(CASE WHEN flagged_6h_before THEN 1 ELSE 0 END) AS flagged_by_model,
           sum(news2_5_before) AS news2_5_or_more
    FROM before
""")
)

These counts cover every scored hour, including the training period, so they look better than the honest **test** results in the model card (23 of 26 vs NEWS2 19 of 26, scored once on hours the model never saw).

## 6. The ward view: KPIs per unit and hour

In [ ]:
display(
    spark.sql(f"""
    SELECT site_id, date(hour_ts) AS day, round(avg(census), 1) AS mean_census,
           sum(alerts_raised) AS alerts, sum(escalations) AS escalations,
           round(sum(alerts_raised) / (avg(census) / 5) / 2, 2) AS alerts_per_nurse_per_shift
    FROM {CATALOG}.gold.site_kpis_hourly GROUP BY ALL ORDER BY day, site_id
""")
)

`alerts_per_nurse_per_shift` assumes one nurse per 5 patients and two 12-hour shifts a day; the budget is about 2.

## 7. Try it yourself

1. Find the hour where the most patients were High at once. Which unit was it, and did any of them escalate?
2. For the patient in section 3, read the `top_factors` at the alert hour. Do they match the vitals in `gold.patient_hour_features`?
3. How many alerts are "not known yet"? Work out from `alert_ts` and the data's last hour why exactly those.
4. Why does each scoring run replace the tables, and what should change once new hours keep arriving (Phase 3)?

In [ ]:
# Try it yourself: 1


In [ ]:
# Try it yourself: 2


**Pitfall: a threshold that drifts with the data.** Fix it on validation at training time; scoring just applies it.

**Pitfall: rescoring history.** Today every run rescores all hours with the current champion. Once the model changes over time, past scores should stay those of the model that was live, or the dashboards rewrite history.

**Next:** a serving endpoint that scores one patient at a time, for the app's what-if.